In [1]:
import pandas as pd

CARTELLA = "/Users/lucaportugalli/Desktop/AIDA/PROJECT WORK/dati/indice_deprivazione_2011/dati_estratti/Sezioni di Censimento"
REGIONI = [f"R{i:02d}" for i in range(1, 21)]
OUTPUT = "sezioni_censimento_2011_italia.csv"


def main():
    frames = []
    for r in REGIONI:
        path = f"{CARTELLA}/{r}_indicatori_2011_sezioni.csv"
        df = pd.read_csv(path, sep=";", encoding="latin1", low_memory=False)
        df = pd.concat({"REGIONE_FILE": pd.Series(r, index=df.index)}, axis=1).join(df)
        frames.append(df)
        print(f"{r}: {len(df):,} sezioni")

    italia = pd.concat(frames, ignore_index=True)
    print(f"\nTotale Italia: {len(italia):,} sezioni, {italia.shape[1]} colonne")

    italia.to_csv(OUTPUT, index=False, encoding="utf-8")
    print(f"Salvato: {OUTPUT}")


if __name__ == "__main__":
    main()

/opt/anaconda3/lib/python3.13/site-packages/pandas/core/computation/expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


R01: 31,830 sezioni
R02: 1,486 sezioni


R03: 48,220 sezioni
R04: 10,010 sezioni


R05: 32,058 sezioni
R06: 7,243 sezioni
R07: 10,441 sezioni


R08: 37,013 sezioni
R09: 27,078 sezioni
R10: 7,092 sezioni


R11: 11,134 sezioni
R12: 29,123 sezioni


R13: 8,280 sezioni
R14: 2,357 sezioni
R15: 22,178 sezioni


R16: 20,627 sezioni
R17: 4,415 sezioni
R18: 11,353 sezioni


R19: 34,064 sezioni
R20: 10,861 sezioni

Totale Italia: 366,863 sezioni, 153 colonne


Salvato: sezioni_censimento_2011_italia.csv


In [2]:
df = pd.read_csv('sezioni_censimento_2011_italia.csv')

In [3]:
df.head()

,REGIONE_FILE,CODREG,REGIONE,CODPRO,PROVINCIA,CODCOM,COMUNE,PROCOM,SEZ2011,NSEZ,...,E22,E23,E24,E25,E26,E27,E28,E29,E30,E31
0,R01,1,Piemonte,1,Torino,1,Agliè,1001,10010000001,1,...,63,26,14,8,1,625,52,193,59,9
1,R01,1,Piemonte,1,Torino,1,Agliè,1001,10010000002,2,...,50,13,6,5,0,375,62,125,25,0
2,R01,1,Piemonte,1,Torino,1,Agliè,1001,10010000006,6,...,1,1,1,0,0,12,1,3,1,0
3,R01,1,Piemonte,1,Torino,1,Agliè,1001,10010000007,7,...,17,9,3,0,0,110,20,36,3,0
4,R01,1,Piemonte,1,Torino,1,Agliè,1001,10010000008,8,...,2,0,0,0,0,25,4,18,1,0


In [4]:
df['P1'].value_counts()

P1
0       17830
1        4730
4        4653
3        4546
2        4378
        ...  
1619        1
1598        1
2013        1
1684        1
1660        1
Name: count, Length: 1956, dtype: int64

In [5]:
# ============================================================
# COSTRUZIONE DATASET RIDOTTO — SEZIONI DI CENSIMENTO 2011
# ============================================================
# Filtriamo le 153 colonne del dataset unito (df) tenendo solo gli
# attributi decisi per il progetto EV Charge Desert, raggruppati per
# motivazione (vedi dizionario_attributi_2011_sezioni.xlsx per le
# definizioni complete di ogni codice).

# 1) TERRITORIO — chiavi identificative e di join: servono per
#    collegare le sezioni a comune/provincia/regione e alle altre
#    fonti del progetto (PUN, POI, parco circolante).
COLONNE_TERRITORIO = [
    "CODREG", "REGIONE", "CODPRO", "PROVINCIA", "CODCOM", "COMUNE",
    "PROCOM", "SEZ2011", "NSEZ", "ACE", "CODLOC", "CODASC",
]

# 2) IDI — le 10 variabili grezze necessarie per calcolare le 4
#    componenti dell'Indice di Deprivazione Italiano ridotto
#    (bassa istruzione, disoccupazione, non proprietà, affollamento).
#    Formula completa in calcola_indice_deprivazione_2011.py.
COLONNE_IDI = [
    "P1",    # popolazione residente totale — filtro sezioni abitate (P1>=1) e peso
    "P46",   # popolazione 6+ anni — denominatore bassa istruzione
    "P49",   # licenza media inferiore — numeratore bassa istruzione
    "P50",   # licenza elementare — numeratore bassa istruzione
    "P60",   # forze lavoro totali — denominatore disoccupazione
    "P62",   # disoccupati in cerca di nuova occupazione — numeratore disoccupazione
    "A44",   # superficie abitazioni occupate da residenti — denominatore affollamento
    "PF2",   # componenti delle famiglie residenti — numeratore affollamento
    "A46",   # famiglie in affitto — numeratore non proprietà
    "PF1",   # famiglie residenti totali — denominatore non proprietà
]

# 3) EDIFICI — epoca di costruzione, numero di piani, numero di
#    interni: proxy della fattibilità di ricarica domestica (edifici
#    bassi/moderni -> più probabile un garage privato con potenza
#    sufficiente per una wallbox; condomini alti e datati -> dipendenza
#    dalla ricarica pubblica).
COLONNE_EDIFICI = [
    "E8", "E9", "E10", "E11", "E12", "E13", "E14", "E15", "E16",  # epoca di costruzione
    "E17", "E18", "E19", "E20",                                   # numero di piani
    "E21", "E22", "E23", "E24", "E25", "E26", "E27",              # numero di interni
]

# 4) REDDITO (proxy) — percettori di reddito da lavoro/capitale:
#    non è un importo di reddito (non esiste a livello di sezione),
#    ma un conteggio di persone con una fonte di reddito, segnale
#    economico complementare alla disoccupazione.
COLONNE_REDDITO = [
    "P139",  # percettori di reddito da lavoro/capitale, 15+ anni, totale
    "P140",  # percettori di reddito da lavoro/capitale, 15+ anni, maschi
]

# 5) SESSO — split maschi/femmine della popolazione totale.
COLONNE_SESSO = [
    "P2",  # popolazione residente maschi
    "P3",  # popolazione residente femmine
]

# 6) ETA' — le fasce quinquennali servono SOLO per calcolare la
#    popolazione in età guida stimata; non vengono mantenute
#    singolarmente nel dataset finale.
#    Criterio: teniamo i bin interi da 20 a 74 anni (P18..P28).
#    Escludiamo:
#      - < 20 anni (P14-P17): minorenni e neopatentati. I neopatentati
#        sono esclusi di proposito perché nel primo anno di patente in
#        Italia vige un limite di potenza/peso (art. 117 Codice della
#        Strada) che molte auto elettriche superano nettamente.
#      - > 74 anni (P29, "età > 74 anni"): coerente con l'esclusione
#        degli over 75.
#    Partendo da un bin intero (20-24) invece di tagliare a metà il
#    bin 15-19, evitiamo di dover stimare una frazione arbitraria di
#    quel bin (nessuna assunzione di distribuzione uniforme necessaria).
COLONNE_ETA_GREZZE = [f"P{i}" for i in range(18, 29)]  # P18 (20-24) ... P28 (70-74)

# ------------------------------------------------------------
# Costruzione del dataframe ridotto
# ------------------------------------------------------------
colonne_da_tenere = (
    COLONNE_TERRITORIO + COLONNE_IDI + COLONNE_EDIFICI
    + COLONNE_REDDITO + COLONNE_SESSO + COLONNE_ETA_GREZZE
)

sezioni_ridotto = df[colonne_da_tenere].copy()

# Popolazione in età guida stimata (20-74 anni): somma dei bin
# quinquennali P18 (20-24) ... P28 (70-74).
sezioni_ridotto["popolazione_eta_guida_stimata"] = sezioni_ridotto[COLONNE_ETA_GREZZE].sum(axis=1)

# I bin grezzi servivano solo a calcolare la somma sopra: li rimuoviamo
# per non duplicare l'informazione nel dataset finale.
sezioni_ridotto = sezioni_ridotto.drop(columns=COLONNE_ETA_GREZZE)

print(f"Dataset ridotto: {sezioni_ridotto.shape[0]:,} sezioni, {sezioni_ridotto.shape[1]} colonne")
print(f"(da {df.shape[1]} colonne originali)")
sezioni_ridotto.head()

Dataset ridotto: 366,863 sezioni, 47 colonne
(da 153 colonne originali)


,CODREG,REGIONE,CODPRO,PROVINCIA,CODCOM,COMUNE,PROCOM,SEZ2011,NSEZ,ACE,...,E23,E24,E25,E26,E27,P139,P140,P2,P3,popolazione_eta_guida_stimata
0,1,Piemonte,1,Torino,1,Agliè,1001,10010000001,1,0,...,26,14,8,1,625,247,95,382,423,543
1,1,Piemonte,1,Torino,1,Agliè,1001,10010000002,2,0,...,13,6,5,0,375,209,96,342,375,522
2,1,Piemonte,1,Torino,1,Agliè,1001,10010000006,6,0,...,1,1,0,0,12,5,3,12,14,22
3,1,Piemonte,1,Torino,1,Agliè,1001,10010000007,7,0,...,9,3,0,0,110,75,39,102,110,144
4,1,Piemonte,1,Torino,1,Agliè,1001,10010000008,8,0,...,0,0,0,0,25,8,3,23,22,33


In [6]:
# Salvataggio del dataset ridotto (47 colonne, vedi cella precedente per
# il criterio di selezione di ciascun gruppo di colonne).
OUTPUT_RIDOTTO = "sezioni_censimento_2011_ridotto.csv"

sezioni_ridotto.to_csv(OUTPUT_RIDOTTO, index=False, encoding="utf-8")
print(f"Salvato: {OUTPUT_RIDOTTO}")
print(f"{sezioni_ridotto.shape[0]:,} righe, {sezioni_ridotto.shape[1]} colonne")

Salvato: sezioni_censimento_2011_ridotto.csv
366,863 righe, 47 colonne
